In [ ]:
# Task 2 — generate_scr_narrative(findings: dict)

In [ ]:
from google import genai
from google.genai import types
from google.colab import userdata

def generate_scr_narrative(findings: dict) -> dict:
    """
    Task 2: Generates an SCR (Situation, Complication, Resolution) narrative report
    for Mamaearth's regional ops and finance heads using the Google GenAI SDK.
    Returns a structured dictionary with status, narrative, and token usage statistics.
    """
    try:
        # 1. Initialize the GenAI Client with a timeout option in seconds (15.0s)
        api_key = userdata.get('HARI_API_KEY')
        client = genai.Client(
            api_key=api_key,
            http_options={'timeout': 15000}
        )

        # 2. Define the strict system instruction
        system_instruction = (
            "You are a senior data analyst writing for Mamaearth's regional ops and finance heads. "
            "Your task is to write a business narrative using the Situation-Complication-Resolution (SCR) framework. "
            "You must structure your response into exactly three labeled sections: 'Situation', 'Complication', and 'Resolution'. "
            "CRITICAL RULE: Every single number, statistic, or metric in your output must come directly from the provided "
            "findings dictionary and appear with the exact same value. You must not invent, extrapolate, "
            "or assume any statistics. If a number is not explicitly provided in the findings, do not include it."
        )

        # 3. Create the user prompt by interpolating the findings dictionary
        user_prompt = f"Here are the quantitative findings from our latest analysis to base your SCR narrative on:\n{findings}"

        # 4. Call the model using explicit configurations
        # temperature=0.0: Set to 0.0 for deterministic output because this is a factual business report.
        # max_output_tokens=500: Hard limit set to ensure a concise narrative.
        config = types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=0.0,
            max_output_tokens=500
        )

        response = client.models.generate_content(
            model='gemini-flash-lite-latest',
            contents=user_prompt,
            config=config
        )

        # Retrieve token counts from the response metadata
        tokens = None
        if response.usage_metadata:
            tokens = {
                "input_tokens": response.usage_metadata.prompt_token_count,
                "output_tokens": response.usage_metadata.candidates_token_count
            }

        return {
            "status": "success",
            "narrative": response.text,
            "tokens": tokens
        }

    except Exception as e:
        # Wrap any exceptions in a structured format instead of letting them crash the flow
        return {
            "status": "error",
            "narrative": None,
            "message": str(e)
        }

In [ ]:
import json

# Load the real findings from the findings.json file in the repository
try:
    with open('/content/findings.json', 'r') as f:
        findings_data = json.load(f)

    # Generate and print the narrative
    narrative_output = generate_scr_narrative(findings_data)
    print(narrative_output)
except FileNotFoundError:
    print("Error: /content/findings.json file not found in the workspace.")
except Exception as e:
    print(f"An error occurred during execution: {e}")

{'status': 'success', 'narrative': "### Situation\nMamaearth's regional operations and finance teams recently conducted a comprehensive revenue and return analysis to establish clear operational baselines. The financial data established a raw total revenue of 99860.2 INR, which reconciled down to a cleaned total revenue of 97358.3 INR after accounting for a duplicate reconciliation delta of 2501.9 INR. Additionally, analysis of temporal performance revealed that the outlier-inflated month of 2026-01 had an apparent revenue of 29582.1 INR, while the true peak month was identified as 2026-03 with a revenue of 20318.9 INR.\n\n### Complication\nDespite the solid baseline revenue figures, regional operations face significant margin erosion driven by high product return rates across payment methods. Specifically, COD orders suffer from a 44.4 return rate, compared to 18.9 for UPI and 14.7 for CARD. This issue is heavily concentrated in specific segments, as the highest risk segment—identifie

In [ ]:
# Task 3 — Parameter locking and error handling execution driver
import json

try:
    # Load actual findings from findings.json
    with open('/content/findings.json', 'r') as f:
        findings_data = json.load(f)

    # Generate and capture structured result using Task 2 function
    print("--- Executing Task 3 (AI Generation with Parameter Locking & Error Handling) ---")
    result = generate_scr_narrative(findings_data)

    # Display output as formatted JSON
    print(json.dumps(result, indent=2))
except Exception as e:
    print(f"An unexpected error occurred in the execution driver: {e}")

--- Executing Task 3 (AI Generation with Parameter Locking & Error Handling) ---
{
  "status": "success",
  "narrative": "### Situation\nOur regional operations and finance teams recently completed a comprehensive revenue and return-rate review for Mamaearth. The analysis established a raw total revenue of 99860.2 INR, which reconciled down by a duplicate reconciliation delta of 2501.9 INR, resulting in a cleaned total revenue of 97358.3 INR. Looking at monthly sales dynamics, the true peak month was recorded as 2026-03, generating 20318.9 INR in revenue. \n\n### Complication\nDespite solid overall numbers, severe distortions and risks threaten our regional efficiency. An outlier-inflated month was identified in 2026-01, where apparent revenue of 29582.1 INR masked the true corrected revenue of 11637.1 INR. Furthermore, payment methods are plagued by disproportionate return rates: COD stands at 44.4%, CARD at 14.7%, and UPI at 18.9%. This issue culminates in our highest risk segment, l

In [ ]:
# Task 4 — Offline fallback path
import json

def generate_scr_narrative_offline(findings: dict) -> dict:
    """
    Fully deterministic fallback function that constructs an SCR narrative report
    locally using Python string interpolation. Guaranteed to work offline with zero config.
    """
    try:
        # Clean extraction of nested structures from findings with fallbacks
        cleaned_rev = findings.get('cleaned_total_revenue_inr', 'N/A')
        raw_rev = findings.get('raw_total_revenue_inr', 'N/A')
        recon_delta = findings.get('duplicate_reconciliation_delta_inr', 'N/A')

        peak_month_info = findings.get('true_peak_month', {})
        peak_month = peak_month_info.get('month', 'N/A')
        peak_rev = peak_month_info.get('revenue_inr', 'N/A')

        returns = findings.get('return_rate_by_payment', {})
        cod_return = returns.get('COD', 'N/A')
        card_return = returns.get('CARD', 'N/A')
        upi_return = returns.get('UPI', 'N/A')

        risk_segment = findings.get('highest_risk_segment', {})
        risk_payment = risk_segment.get('payment_method', 'N/A')
        risk_tier = risk_segment.get('city_tier', 'N/A')
        risk_rate = risk_segment.get('return_rate_pct', 'N/A')

        outlier_info = findings.get('outlier_inflated_month', {})
        outlier_month = outlier_info.get('month', 'N/A')
        apparent_rev = outlier_info.get('apparent_revenue_inr', 'N/A')
        corrected_rev = outlier_info.get('corrected_revenue_inr', 'N/A')

        narrative = (
            "Situation:\n"
            f"Mamaearth's latest financial cycle recorded a raw total revenue of {raw_rev} INR. "
            f"Following thorough analysis and reconciling a duplicate delta of {recon_delta} INR, "
            f"the true cleaned total revenue is established at {cleaned_rev} INR. "
            f"Performance metrics verify {peak_month} as the true peak month, generating {peak_rev} INR.\n\n"
            "Complication:\n"
            f"Profitability remains severely constrained by high logistics return rates across payment platforms, "
            f"specifically marking a return rate of {cod_return}% for COD, {card_return}% for CARD, and {upi_return}% for UPI. "
            f"Concentrated risk is highest in {risk_payment} orders within City Tier {risk_tier}, "
            f"suffering a peak return rate of {risk_rate}%. Operational planning was further complicated by "
            f"reporting distortions during {outlier_month}, when an outlier inflated apparent revenue to "
            f"{apparent_rev} INR before corrective adjustments stabilized it at {corrected_rev} INR.\n\n"
            "Resolution:\n"
            f"To optimize regional margins, operations must enforce strict verification controls on high-risk "
            f"{risk_payment} orders in City Tier {risk_tier} to address the critical {risk_rate}% return rate. "
            f"Finance heads must align planning baselines around the verified cleaned total revenue of {cleaned_rev} INR "
            f"and deploy systemic anomaly safeguards to prevent uncorrected spikes like the {apparent_rev} INR distortion."
        )

        return {
            "status": "success",
            "narrative": narrative,
            "tokens": None
        }
    except Exception as e:
        return {
            "status": "error",
            "narrative": None,
            "message": f"Offline fallback generator error: {str(e)}"
        }

try:
    # Load the actual findings data
    with open('/content/findings.json', 'r') as f:
        findings_data = json.load(f)

    # Test offline generation by executing the local fallback generator
    print("--- Executing Fallback Generator (Deterministic Offline Mode) ---")
    offline_result = generate_scr_narrative_offline(findings_data)
    print(json.dumps(offline_result, indent=2))

except Exception as e:
    print(f"An error occurred in Task 4 testing: {e}")

--- Executing Fallback Generator (Deterministic Offline Mode) ---
{
  "status": "success",
  "narrative": "Situation:\nMamaearth's latest financial cycle recorded a raw total revenue of 99860.2 INR. Following thorough analysis and reconciling a duplicate delta of 2501.9 INR, the true cleaned total revenue is established at 97358.3 INR. Performance metrics verify 2026-03 as the true peak month, generating 20318.9 INR.\n\nComplication:\nProfitability remains severely constrained by high logistics return rates across payment platforms, specifically marking a return rate of 44.4% for COD, 14.7% for CARD, and 18.9% for UPI. Concentrated risk is highest in COD orders within City Tier 2, suffering a peak return rate of 54.5%. Operational planning was further complicated by reporting distortions during 2026-01, when an outlier inflated apparent revenue to 29582.1 INR before corrective adjustments stabilized it at 11637.1 INR.\n\nResolution:\nTo optimize regional margins, operations must enforc

In [ ]:
# Task 5 — Numeric accuracy checklist
import json
import re

def verify_narrative_accuracy(file_path: str):
    """
    Validates that the generated business narrative contains all five key
    quantitative metrics, printing a clear PASS/FAIL checklist status for each.
    """
    print(f"=== Starting Numeric Accuracy Verification on: {file_path} ===")
    try:
        with open(file_path, 'r') as f:
            content = f.read()

        # Try to parse as JSON if it was stored as a structured dictionary
        try:
            data = json.loads(content)
            narrative_text = data.get("narrative", "")
        except json.JSONDecodeError:
            narrative_text = content

        # Normalize the narrative text: convert to lowercase and remove commas from numbers (e.g., 97,358.30 -> 97358.30)
        normalized_text = narrative_text.lower()
        normalized_text = re.sub(r'(\d),(\d)', r'\1\2', normalized_text)

        # Helper checking function that handles float matches (e.g., matching both 97358.3 and 97358.30)
        def contains_metric(target_val):
            if isinstance(target_val, float):
                # Match the float cleanly or with trailing zero
                pattern_short = f"{target_val:.1f}"
                pattern_long = f"{target_val:.2f}"
                return (pattern_short in normalized_text) or (pattern_long in normalized_text)
            return str(target_val).lower() in normalized_text

        # Checklist items
        checklist = [
            {
                "name": "Cleaned Total Revenue (97358.3)",
                "check": contains_metric(97358.3)
            },
            {
                "name": "COD Return Rate (44.4)",
                "check": contains_metric(44.4)
            },
            {
                "name": "COD + Tier-2 Return Rate (54.5)",
                "check": contains_metric(54.5)
            },
            {
                "name": "Reconciliation Delta (2501.9)",
                "check": contains_metric(2501.9)
            },
            {
                "name": "True Peak Month (March / 2026-03 and 20318.9)",
                "check": (contains_metric("march") or contains_metric("2026-03")) and contains_metric(20318.9)
            }
        ]

        passed_all = True
        for item in checklist:
            status = "[PASS]" if item["check"] else "[FAIL]"
            if not item["check"]:
                passed_all = False
            print(f"{status} - {item['name']}")

        print("====================================================")
        if passed_all:
            print("SUCCESS: All numeric metrics successfully validated in the narrative!")
        else:
            print("WARNING: Some numeric checks failed. Please verify narrative text.")

    except Exception as e:
        print(f"Verification failed with an error: {e}")

# Execute the checker against /content/sample_output.txt
verify_narrative_accuracy('/content/sample_output.txt')

=== Starting Numeric Accuracy Verification on: /content/sample_output.txt ===
[PASS] - Cleaned Total Revenue (97358.3)
[PASS] - COD Return Rate (44.4)
[PASS] - COD + Tier-2 Return Rate (54.5)
[PASS] - Reconciliation Delta (2501.9)
[PASS] - True Peak Month (March / 2026-03 and 20318.9)
SUCCESS: All numeric metrics successfully validated in the narrative!
